You're working as a sports journalist at a major online sports media company, specializing in soccer analysis and reporting. You've been watching both men's and women's international soccer matches for a number of years, and your gut instinct tells you that more goals are scored in women's international football matches than men's. This would make an interesting investigative article that your subscribers are bound to love, but you'll need to perform a valid statistical hypothesis test to be sure!

While scoping this project, you acknowledge that the sport has changed a lot over the years, and performances likely vary a lot depending on the tournament, so you decide to limit the data used in the analysis to only official FIFA World Cup matches (not including qualifiers) since 2002-01-01.

You create two datasets containing the results of every official men's and women's international football match since the 19th century, which you scraped from a reliable online source. This data is stored in two CSV files: women_results.csv and men_results.csv.

The question you are trying to determine the answer to is:

Are more goals scored in women's international soccer matches than men's?

You assume a 10% significance level, and use the following null and alternative hypotheses:

 : The mean number of goals scored in women's international soccer matches is the same as men's.

 : The mean number of goals scored in women's international soccer matches is greater than men's.

In [ ]:
# Start your code here!
import pandas as pd
from scipy import stats
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind
import pingouin
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
df_women = pd.read_csv('women_results.csv')
df_men = pd.read_csv('men_results.csv')
df_women['date'] = pd.to_datetime(df_women['date'],format= '%Y-%m-%d')
df_men['date'] = pd.to_datetime(df_men['date'],format= '%Y-%m-%d')
df_women['gender'] = 'female'
df_men['gender'] = 'male'

df_data = pd.concat([df_women,df_men])

In [ ]:
df_data = df_data[df_data['tournament'] == 'FIFA World Cup']
df_data = df_data[df_data['date'] >= '2002-01-01']
df_data['goals'] = df_data['home_score'] + df_data['away_score']

In [ ]:
print(df_data.info())

In [ ]:
mean_response = df_data.pivot_table(
  values='goals', index='gender', aggfunc="mean")
print(f"The mean goals scored by each gender are: \n\n {mean_response}" )

In [ ]:
sns.boxplot(data=df_data,x='gender',y='goals')
plt.show()
sns.histplot(data=df_data,hue='gender',x='goals',multiple='dodge',stat='percent',common_norm=False)
plt.show()

In [ ]:
# we determine there is a likely difference in the number of goals between female and male football games: that is there are more goals scored in female matches
# we also determine that the data may not be considered normal
# so let's test this formally!

gender_types = ['female', 'male']
# do a list comprehension to form a list of groups
for gender in gender_types:
    group = df_data [ df_data ['gender' ] == gender ]['goals']
    
    # Perform D'Agostino's K^2 test
    statistic, pvalue = stats.normaltest(group)
    print(f"For {gender} games, D'Agostino's K^2 statistic is: {statistic: .3f}, pvalue: {pvalue: .3f}")


In [ ]:
# we conclude that the distributions for both male and female football games are non-normal

# two approaches possible:
# 1) do a transform
# 2) do a non-parametric test

# I will elect for a non-parametric test, I speculate that a transform would be deleterious to discrete and low numbers data

# select Mann Whitney U test

df_females = df_data[df_data['gender'] == 'female']['goals']
df_males = df_data[df_data['gender'] == 'male']['goals']

u_stat, u_pval = mannwhitneyu(
	df_females,
    df_males,
    alternative="greater"
)
print (f"Mann-Whitney U test p-value: {u_pval: .4f}")

In [ ]:
result_dict = {"p_val": u_pval, "result": "reject"}

print(result_dict)